# How to save and load grid files

Import from file, convert to something else

Note that geopandas is not needed here.

## Code setup

In [1]:
# Setting up paths to files:
import os
from dataclasses import dataclass

# Loading files:
import numpy as np  # to load grids directly
import pandas as pd  # to load props dataframe
import json  # to load dict
import geojson  # to load outlines

# Add path to package:
import sys
sys.path.insert(0,'../')
import isopods.outline
import isopods.convert

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    dir_data: str = '../data'
    # Always need this dict:
    dict_filename: str = 'lsoa2011_devon_1k_grid_dict.json'
    # Option 1: load in this dataframe
    grid_filename: str = 'lsoa2011_devon_1k_grid_df.csv'
    # Option 2: load in these grids
    lsoa_grid_data_filename: str = 'lsoa2011_devon_1k_grid_data.csv'
    lsoa_grid_mask_filename: str = 'lsoa2011_devon_1k_grid_mask.csv'
    # Polygon outlines:
    outline_filename: str = 'districts2011_devon_1k_outlines.geojson'
    # Polygon midpoints:
    midpoints_filename: str = 'districts2011_devon_1k_midpoints.csv'

paths = Paths()

## Transform dictionary

Load:

In [3]:
p = os.path.join(paths.dir_data, paths.dict_filename)
with open(p, 'r') as f:
    transform_dict = json.load(f)

In [4]:
transform_dict

{'xmin': 212000.0,
 'ymax': 152000.0,
 'cell_width': 1000,
 'cell_height': 1000,
 'n_cols': 126,
 'n_rows': 118,
 'xmax': 338000.0,
 'ymin': 34000.0}

Save:

## Grid from dataframe

Load:

In [5]:
p = os.path.join(paths.dir_data, paths.grid_filename)
# Use converters to turn proportion data from strings to lists.
converters = {
    'label_props': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
    'area_props': lambda x: [float(f) for f in x[1:-1].split()]
}
df_props = pd.read_csv(p, index_col=0, converters=converters)

In [6]:
df_props.head(3)

,cell_row,cell_col,label_props,area_props,prop_cell_covered,label_majority,overlapping_polys
cell_label,,,,,,,
109,109,0,[E01020288],[1.0],0.000,E01020288,0
110,110,0,[E01020288],[1.0],0.124,E01020288,0
111,111,0,[E01020288],[1.0],0.089,E01020288,0


Convert to array using the transform dict:

In [7]:
# Only keep data where the cell is mostly covered:
mask = df_props['prop_cell_covered'] >= 0.5
# Make grid of region labels then convert to required data.
arr_labels = isopods.convert.convert_df_to_2darray(
    df_props.loc[mask], 'label_majority', **transform_dict)

In [8]:
arr_labels[42:45, 20:30]

masked_array(
  data=[[--, --, --, --, --, --, --, 'E01020325', 'E01020325',
         'E01020331'],
        [--, --, --, --, --, --, --, 'E01020325', 'E01020325',
         'E01020325'],
        [--, --, --, --, --, 'E01020325', 'E01020325', 'E01020325',
         'E01020325', 'E01020325']],
  mask=[[ True,  True,  True,  True,  True,  True,  True, False, False,
         False],
        [ True,  True,  True,  True,  True,  True,  True, False, False,
         False],
        [ True,  True,  True,  True,  True, False, False, False, False,
         False]],
  fill_value=np.str_('?'),
  dtype=object)

Save:

## Load in array directly

### Masked array

If there is no mask data, then load in only the data and use a standard numpy array instead of a masked array.

In [9]:
p = os.path.join(paths.dir_data, paths.lsoa_grid_data_filename)
lsoa_grid_data = np.loadtxt(p, dtype=str, delimiter=',')
p = os.path.join(paths.dir_data, paths.lsoa_grid_mask_filename)
lsoa_grid_mask = np.loadtxt(p, dtype=bool, delimiter=',')

arr_labels = np.ma.MaskedArray(
    lsoa_grid_data, mask=lsoa_grid_mask, dtype=object)

Preview a few rows of data:

In [10]:
arr_labels[42:45, 20:30]

masked_array(
  data=[[--, --, --, --, --, --, 'E01020325', 'E01020325', 'E01020325',
         'E01020331'],
        [--, --, --, --, 'E01020325', 'E01020325', 'E01020325',
         'E01020325', 'E01020325', 'E01020325'],
        [--, --, --, --, 'E01020325', 'E01020325', 'E01020325',
         'E01020325', 'E01020325', 'E01020325']],
  mask=[[ True,  True,  True,  True,  True,  True, False, False, False,
         False],
        [ True,  True,  True,  True, False, False, False, False, False,
         False],
        [ True,  True,  True,  True, False, False, False, False, False,
         False]],
  fill_value=np.str_('?'),
  dtype=object)

Save.

Note that you have to specify the data format. For example, region names are strings '%s', integers '%i', float '%f'...

Use comma as delimiter for csv.

### Standard array

Load in:

In [11]:
p = os.path.join(paths.dir_data, paths.lsoa_grid_data_filename)
arr_labels = np.loadtxt(p, dtype=str, delimiter=',')

Preview a few rows of data:

In [12]:
arr_labels[42:45, 20:30]

array([['', '', '', '', '', '', 'E01020325', 'E01020325', 'E01020325',
        'E01020331'],
       ['', '', '', '', 'E01020325', 'E01020325', 'E01020325',
        'E01020325', 'E01020325', 'E01020325'],
       ['', '', '', '', 'E01020325', 'E01020325', 'E01020325',
        'E01020325', 'E01020325', 'E01020325']], dtype='<U9')

Save.

Note that you have to specify the data format. For example, region names are strings '%s', integers '%i', float '%f'...

Use comma as delimiter for csv.

## Outline polygons

Load in:

In [13]:
p = os.path.join(paths.dir_data, paths.outline_filename)
with open(p, 'r') as f:
    gj = geojson.load(f)

dict_polys = isopods.outline.convert_featurecollection_to_dict_polys(gj)

In [14]:
dict_polys

{'East Devon': <MULTIPOLYGON (((294000 102000, 294000 104000, 295000 104000, 295000 102000,...>,
 'Exeter': <POLYGON ((296000 87000, 296000 89000, 291000 89000, 291000 91000, 290000 91...>,
 'Mid Devon': <POLYGON ((277000 91000, 277000 92000, 276000 92000, 276000 93000, 273000 93...>,
 'North Devon': <POLYGON ((276000 110000, 276000 112000, 275000 112000, 275000 113000, 27300...>,
 'Plymouth': <MULTIPOLYGON (((246000 50000, 246000 51000, 248000 51000, 248000 50000, 246...>,
 'South Hams': <POLYGON ((277000 34000, 277000 35000, 276000 35000, 276000 36000, 273000 36...>,
 'Teignbridge': <POLYGON ((274000 64000, 274000 65000, 273000 65000, 273000 72000, 272000 72...>,
 'Torbay': <POLYGON ((289000 54000, 289000 56000, 288000 56000, 288000 58000, 286000 58...>,
 'Torridge': <MULTIPOLYGON (((234000 85000, 234000 90000, 233000 90000, 233000 91000, 232...>,
 'West Devon': <POLYGON ((244000 61000, 244000 62000, 243000 62000, 243000 64000, 241000 64...>}

Save:

## Region midpoints

In [15]:
p = os.path.join(paths.dir_data, paths.midpoints_filename)
df_meds = pd.read_csv(p, index_col='region')

In [16]:
df_meds.head(2)

,x_mid,y_mid
region,,
East Devon,312000.0,96000.0
Exeter,293000.0,92000.0


Save: